In [ ]:
model = "Mistral-Nemo-Instruct-2407" # nicht Qwen3-8B weil reasoning
embeddings_output = [f"embeddings_{model}_Teil1_eigenanteil.json", f"embeddings_{model}_Teil2_eigenanteil.json",f"embeddings_{model}_Teil3_eigenanteil.json",
                     f"embeddings_{model}_Teil4_eigenanteil.json", 
                     f"embeddings_{model}_Teil5_eigenanteil.json"
                     ]
answers = [
    f"../Antworten/answers_{model}_Teil1_eigenanteil.txt", f"../Antworten/answers_{model}_Teil2_eigenanteil.txt", f"../Antworten/answers_{model}_Teil3_eigenanteil.txt",
    f"../Antworten/answers_{model}_Teil4_eigenanteil.txt",
    f"../Antworten/answers_{model}_Teil5_eigenanteil.txt"
    ]


In [12]:
import os
import re
import time
import json
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

SAMPLES_PER_QUESTION = 10

for n, answers in enumerate(answers):
    with open(answers, "r", encoding="utf-8") as f:
        text = f.read()

    # Alle Texte direkt nach "newOutput:" extrahieren
    outputs = [
        match.strip()
        for match in re.findall(r"newOutput:\s*(.*?)(?=\n\s*newOutput:|\Z)", text, flags=re.DOTALL)
        if match.strip()
    ]

    """ if not outputs:
        raise ValueError(f"Keine Inhalte nach 'newOutput:' in {answers} gefunden.") """

    num_questions = len(outputs) // SAMPLES_PER_QUESTION
    out = [["" for _ in range(SAMPLES_PER_QUESTION)] for _ in range(num_questions)]

    for i in range(num_questions):
        for j in range(SAMPLES_PER_QUESTION):
            idx = i * SAMPLES_PER_QUESTION + j
            if idx < len(outputs):
                out[i][j] = outputs[idx]

    load_dotenv(find_dotenv(usecwd=True))
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY fehlt. Bitte in der .env setzen.")

    client = OpenAI(api_key=api_key)

    # gleiche Struktur wie out: embeddings_2d[i][j] gehört zu out[i][j]
    embeddings_2d = [[None for _ in row] for row in out]

    for i, row in enumerate(out):
        for j, generated_text in enumerate(row):
            if not isinstance(generated_text, str) or not generated_text.strip():
                continue
            #print(generated_text)
            response = client.embeddings.create(
                input=generated_text,
                model="text-embedding-3-small"
            )
            embeddings_2d[i][j] = response.data[0].embedding

            # etwas langsamer senden, um Rate-Limits zu vermeiden
            time.sleep(0.05)

    print(f"Extrahierte newOutput-Texte: {len(outputs)}")
    print(f"Fertig. Anzahl Zeilen: {len(embeddings_2d)}, Spalten (Zeile 0): {len(embeddings_2d[0]) if embeddings_2d else 0}")
    print(f"Beispiel-Vektor-Länge [0][0]: {len(embeddings_2d[0][0]) if embeddings_2d and embeddings_2d[0][0] is not None else 'kein Wert'}")

    repo_root = Path(find_dotenv(usecwd=True)).resolve().parent
    output_path = repo_root / "Embeddings" / embeddings_output[n]

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(embeddings_2d, f, ensure_ascii=False)

    print(f"Embeddings gespeichert unter: {output_path}")

Extrahierte newOutput-Texte: 1000
Fertig. Anzahl Zeilen: 100, Spalten (Zeile 0): 10
Beispiel-Vektor-Länge [0][0]: 1536
Embeddings gespeichert unter: /home/max/Dokumente/Master_WI/3.Semester/Data Mining/replikation-data-mining/Embeddings/embeddings_Mistral-Nemo-Instruct-2407_Teil4_eigenanteil.json
Extrahierte newOutput-Texte: 1000
Fertig. Anzahl Zeilen: 100, Spalten (Zeile 0): 10
Beispiel-Vektor-Länge [0][0]: 1536
Embeddings gespeichert unter: /home/max/Dokumente/Master_WI/3.Semester/Data Mining/replikation-data-mining/Embeddings/embeddings_Mistral-Nemo-Instruct-2407_Teil5_eigenanteil.json
